# Pré-Processo
O dataset é muito grande para ser versionado no github, para realizar o download do mesmo através das fontes originais, executar a célula abaixo.

In [ ]:
import requests
import tarfile
import os

URL = "http://mtg.upf.edu/static/datasets/last.fm/lastfm-dataset-360K.tar.gz"
DEST = "data/raw"

os.makedirs(DEST, exist_ok=True)
caminho_tar = os.path.join(DEST, "lastfm-dataset-360K.tar.gz")

headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
resp = requests.get(URL, headers=headers, stream=True, timeout=60)
resp.raise_for_status()

with open(caminho_tar, "wb") as f:
    for chunk in resp.iter_content(chunk_size=8192):
        f.write(chunk)

with tarfile.open(caminho_tar) as tar:
    tar.extractall(DEST)

print("Download concluído.")

# Análise Exploratória — Last.fm-360K
Base composta por dois arquivos: **plays** (interações usuário-artista) e **profile** (dados demográficos dos usuários).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

CAMINHO_PLAYS = "data/raw/usersha1-artmbid-artname-plays.tsv"
CAMINHO_PROFILE = "data/raw/usersha1-profile.tsv"

## 1. Carregamento dos dados

In [ ]:
cols_plays = ["user_id", "artist_mbid", "artist_name", "plays"]
df_plays = pd.read_csv(CAMINHO_PLAYS, sep="\t", names=cols_plays, header=None)

cols_profile = ["user_id", "gender", "age", "country", "signup_date"]
df_profile = pd.read_csv(CAMINHO_PROFILE, sep="\t", names=cols_profile, header=None)

df_plays.head()
df_profile.head();

## 2. PLAYS — visão geral e valores nulos

In [ ]:
df_plays.info()
print(df_plays.describe())
print("\nValores nulos por coluna:")
print(df_plays.isna().sum())

## 3. PLAYS — contagem de usuários e artistas únicos

In [ ]:
n_usuarios = df_plays["user_id"].nunique()
n_artistas = df_plays["artist_name"].nunique()

print(f"Usuários únicos: {n_usuarios}")
print(f"Artistas únicos: {n_artistas}")
print(f"Total de interações: {len(df_plays)}")

## 4. PLAYS — distribuição de reproduções por usuário
Cauda longa: poucos usuários com muitas reproduções, muitos com poucas.

In [ ]:
plays_por_usuario = df_plays.groupby("user_id")["plays"].sum()
print(plays_por_usuario.describe())

plays_por_usuario.hist(bins=50, log=True)
plt.xlabel("Total de reproduções")
plt.ylabel("Nº de usuários (escala log)")
plt.title("Distribuição de reproduções por usuário")
plt.show()

## 5. PLAYS — top 20 artistas por total de reproduções

In [ ]:
top_artistas = (
    df_plays.groupby("artist_name")["plays"]
    .sum()
    .sort_values(ascending=False)
    .head(20)
)
top_artistas

## 6. PLAYS — esparsidade da matriz usuário-artista

In [ ]:
esparsidade = 1 - len(df_plays) / (n_usuarios * n_artistas)
print(f"Esparsidade da matriz usuário-artista: {esparsidade:.6f}")

## 7. PROFILE — visão geral e valores nulos

In [ ]:
df_profile.info()
print("\nValores nulos por coluna:")
print(df_profile.isna().sum())

## 8. PROFILE — gênero

In [ ]:
df_profile["gender"].value_counts(dropna=False)

## 9. PROFILE — idade
filtramos antes de plotar, deixando apenas valores 0 < idade < 100

In [ ]:
print("Idade (bruta, sem filtro):")
print(df_profile["age"].describe())

idade_valida = df_profile[(df_profile["age"] > 0) & (df_profile["age"] < 100)]["age"]
print("\nIdade (0 < idade < 100):")
print(idade_valida.describe())

idade_valida.hist(bins=30)
plt.xlabel("Idade")
plt.ylabel("Nº de usuários")
plt.title("Distribuição de idade (valores válidos)")
plt.show()

## 10. PROFILE — top 20 países

In [ ]:
df_profile["country"].value_counts(dropna=False).head(20)

## 11. PROFILE — intervalo de datas de cadastro

In [ ]:
df_profile["signup_date_parsed"] = pd.to_datetime(df_profile["signup_date"], errors="coerce")
print(df_profile["signup_date_parsed"].min(), "→", df_profile["signup_date_parsed"].max())

## 12. Cruzamento — volume de interação × perfil

In [ ]:
df_merge = df_profile.merge(
    plays_por_usuario.rename("total_plays"), on="user_id", how="left"
)

print("Total de reproduções (mediana) por país — top 10:")
print(df_merge.groupby("country")["total_plays"].median().sort_values(ascending=False).head(10))

print("\nUsuários no profile sem nenhuma interação em plays:")
print(df_merge["total_plays"].isna().sum())